## Setup

In [1]:
# set libraries to refresh
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import contextily as ctx
import xyzservices as xyz
import matplotlib.pyplot as plt

# import kml reading and set supported driver
import fiona
fiona.drvsupport.supported_drivers["KML"] = "rw"

# for parsing HTML inside the Description field
from bs4 import BeautifulSoup

# for TIFF files
import rasterio
from rasterio.plot import show
from rasterstats import zonal_stats
from rasterio.features import geometry_mask

In [3]:
from gridsample.utils import create_ids, save_shapefiles

In [4]:
ROOT_DIR = Path("../")
DATA_DIR = ROOT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "00_raw"
PROCESSED_DATA_DIR = DATA_DIR / "01_processed" / "Solar Parks"

## Load raw shapes and process

### Dhar

In [5]:
# Load dhar181224
raw_dhar_gdf = gpd.read_file(
    RAW_DATA_DIR / "solar_park_shapefiles" / "dhar181224" / "doc.kml",
    driver="KML"
)

In [6]:
# remove z-dimension
raw_dhar_gdf.geometry = raw_dhar_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)
# break up multi-polygons and only keep the polygon
raw_dhar_gdf = raw_dhar_gdf.explode(index_parts=False)
raw_dhar_gdf = raw_dhar_gdf[raw_dhar_gdf.geometry.type == "Polygon"]

# remove useless Description column
dhar_gdf = raw_dhar_gdf.drop(columns="Description")

In [7]:
# drop large green shapes (open .KMZ on Google Earth to see)
dhar_gdf = dhar_gdf[dhar_gdf["Name"] != ""]

In [8]:
# clean up Name so we can separate the villages (string names) from the areas (numbers only)
dhar_gdf["cleaned_name"] = [
    value[0].strip() for value in dhar_gdf["Name"].str.split("/")
]
dhar_gdf["cleaned_name"] = [
    value[0].strip() for value in dhar_gdf["cleaned_name"].str.split(",")
]
# manual clean
dhar_gdf.loc[dhar_gdf["Name"] == "2829Z1", "cleaned_name"] = "2829"

In [ ]:
# ISOLATE AREA ONLY - select rows that have a number as their Name
dhar_yellow_shapes_gdf = dhar_gdf[dhar_gdf["cleaned_name"].str.isnumeric()]
dhar_yellow_shapes_gdf.plot()

In [ ]:
# ISOLATE VILLAGES ONLY - select rows that have a string as their Name
dhar_village_shapes_gdf = dhar_gdf[~dhar_gdf["cleaned_name"].str.isnumeric()]
dhar_village_shapes_gdf = dhar_village_shapes_gdf.drop(columns="cleaned_name")
dhar_village_shapes_gdf = dhar_village_shapes_gdf.rename(
    columns={"Name": "village_name"}
)
dhar_village_shapes_gdf.plot(column="village_name")

In [ ]:
# add village names to areas
dhar_processed_areas_gdf = dhar_yellow_shapes_gdf.sjoin(
    dhar_village_shapes_gdf, how="left", predicate="intersects"
).drop(columns="index_right")
dhar_processed_areas_gdf.plot(column="village_name")
print("Missing village name:", dhar_processed_areas_gdf["village_name"].isnull().sum())
print("Has village name:", dhar_processed_areas_gdf["village_name"].notnull().sum())

In [12]:
# save to file
save_shapefiles(
    dhar_processed_areas_gdf,
    PROCESSED_DATA_DIR / "processed_areas",
    "dhar_processed_all",
    formats=["kml", "parquet"],
)

### Sagar

In [13]:
# ### Unzip KMZ to get KML - USE GOOGLE EARTH TO DO THIS CORRECTLY
# import zipfile

# # Extract the .kml file from the .kmz file
# with zipfile.ZipFile(RAW_DATA_DIR / "solar_park_shapefiles" / "sagar220724.kmz", 'r') as kmz:
#     kmz.extractall(RAW_DATA_DIR / "solar_park_shapefiles" / "sagar220724" )

In [14]:
gdfs = []

for filename in ["sagar_khamkuwa", "sagar_mokalpur", "sagar_tekapar"]:
    gdf = gpd.read_file(
        RAW_DATA_DIR / "solar_park_shapefiles" / "Google Earth" / f"{filename}.kml",
        driver="KML",
    )
    gdf["source"] = filename
    gdfs.append(gdf)

raw_sagar_gdf = pd.concat(gdfs, ignore_index=True)

In [ ]:
# remove z-dimension
raw_sagar_gdf.geometry = raw_sagar_gdf.geometry.apply(
    lambda x: shapely.wkb.loads(shapely.wkb.dumps(x, output_dimension=2))
)
# break up multi-polygons (each one just contains one polygon anyway)
raw_sagar_gdf = raw_sagar_gdf.explode(column='geometry')
raw_sagar_gdf.plot(column="source", legend=True)

In [16]:
# Parse description variables

def description_parser(html_content):
    # Parse the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    # Find the inner table containing the attributes
    inner_table = soup.find_all('table')[1]

    # Extract rows from the inner table
    rows = inner_table.find_all('tr')

    # Create a dictionary to store attributes and their values
    data = {}
    for row in rows:
        cols = row.find_all('td')
        if len(cols) == 2:
            key = cols[0].text.strip()
            value = cols[1].text.strip()
            data[key] = value

    return pd.DataFrame([data])

In [17]:
# make dataframe of variables
data = [description_parser(raw_sagar_gdf["Description"].values[i]) for i in range(len(raw_sagar_gdf))]
df = pd.concat(data)
df.set_index(raw_sagar_gdf.index, inplace=True)

In [18]:
# merge with shapes
raw_sagar_gdf.drop(columns=["Name", "Description"], inplace=True)
sagar_gdf = raw_sagar_gdf.merge(df, left_index=True, right_index=True)

In [ ]:
sagar_gdf.plot(column="PAR_TYPE")

In [20]:
save_shapefiles(
    sagar_gdf,
    PROCESSED_DATA_DIR / "processed_areas",
    "sagar_processed_all",
    formats=["kml", "parquet"],
)

In [21]:
# # get unique values for all columns
# for col in sagar_gdf.columns:
#     print(f"Column: {col}")
#     print(sagar_gdf[col].unique())
#     print("\n")

## Load processed shapes

In [5]:
dhar_processed_areas_gdf = gpd.read_parquet(PROCESSED_DATA_DIR / "processed_areas" / "dhar_processed_all.parquet")
dhar_processed_areas_gdf["source"] = "dhar"

In [6]:
dhar_processed_areas_gdf["parcel_id"] = "DHAR_" + dhar_processed_areas_gdf["Name"]

# Function to add suffix to duplicates
def add_suffix_to_duplicates(series):
    counts = {}
    result = []
    for item in series:
        if item in counts:
            counts[item] += 1
            result.append(f"{item}_{counts[item]}")
        else:
            counts[item] = 0
            result.append(item)
    return result

# Apply the function to the parcel_id column
dhar_processed_areas_gdf["parcel_id"] = add_suffix_to_duplicates(dhar_processed_areas_gdf["parcel_id"])

In [7]:
sagar_gdf = gpd.read_parquet(PROCESSED_DATA_DIR / "processed_areas" / "sagar_processed_all.parquet")

In [8]:
sagar_gdf["parcel_id"] = "SAGAR_" + sagar_gdf["UNQID"]

In [9]:
# filter to only the "PA" PAR_TYPE (since it looks like the barren land)
sagar_gdf = sagar_gdf[sagar_gdf["PAR_TYPE"] == "PA"]

In [10]:
gdf = pd.concat([dhar_processed_areas_gdf, sagar_gdf], ignore_index=True)

In [ ]:
gdf.plot()

In [12]:
gdf = gdf[["source", "parcel_id", "UNQID", "Name", "geometry"]]

In [13]:
gdf = gdf.rename(columns={"UNQID": "SAGAR_UNQID", "Name": "DHAR_Name"})

In [14]:
gdf["Parcel Area (m2)"] = gdf["geometry"].to_crs("24378").area

In [ ]:
gdf.head()

## External data sources

### Buildings

In [16]:
from s2cell.s2cell import lat_lon_to_cell_id
import boto3

#### Download rooftop data

Get the ID of the level 6 S2 Cell that this area sits inside

In [17]:
s2_ids = []

for index, row in gdf.iterrows():
    lat = row["geometry"].centroid.y
    lon = row["geometry"].centroid.x
    s2_cell_id = lat_lon_to_cell_id(lat=lat, lon=lon, level=6)
        
    s2_ids.append(s2_cell_id)

s2_ids = list(set(s2_ids))


Download closest S2 cell shapefile from https://beta.source.coop/vida/google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/

In [ ]:
for s2_cell_id in s2_ids:
    s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"

    if s2_rooftops_path.exists():
        print("File already exists")
    else:
        s3 = boto3.client("s3", endpoint_url="https://data.source.coop")
        s3.download_file(
            "vida",
            f"google-microsoft-open-buildings/geoparquet/by_country_s2/country_iso=IND/{s2_cell_id}.parquet",
            str(s2_rooftops_path),
        )
        print("File downloaded.")

#### Load and process rooftop data

In [19]:
rooftop_gdf_list = []
for s2_cell_id in s2_ids:

    s2_rooftops_path = RAW_DATA_DIR / "rooftops" / f"{s2_cell_id}.parquet"
    rooftop_gdf = gpd.read_parquet(s2_rooftops_path)
    rooftop_gdf_list.append(rooftop_gdf)

rooftop_gdf = pd.concat(rooftop_gdf_list, ignore_index=True)

In [20]:
rooftop_gdf = rooftop_gdf.drop(columns=["boundary_id", "s2_id", "geohash", "bbox", "country_iso"])

In [21]:
rooftop_gdf["rooftop_id"] = create_ids(len(rooftop_gdf), f"ROOFTOP_S2_{s2_cell_id}_")

In [ ]:
rooftop_gdf

#### Match rooftops to colonies

In [23]:
subset_rooftops_gdf = rooftop_gdf.sjoin(gdf, how="inner", predicate="intersects").drop(columns=["index_right"])

#### Produce plots and save to file

In [24]:
for source in gdf["source"].unique():
    folder_path = PROCESSED_DATA_DIR / source
    folder_path.mkdir(parents=True, exist_ok=True)

    area_rooftops = subset_rooftops_gdf[subset_rooftops_gdf["source"] == source]
    ax = gdf[gdf["source"] == source].plot(color="orange", alpha=0.2, figsize=(10, 15))

    # plot buildings
    area_rooftops.plot(ax=ax)
    # ctx.add_basemap(ax, crs=subset_rooftops_gdf.crs.to_string(), source=xyz.TileProvider.from_qms("Google Satellite Hybrid"))
    ax.set_axis_off()
    plt.title(f"{source} Rooftops")
    
    plt.savefig(folder_path / "rooftops.png", bbox_inches="tight")
    plt.close()

    save_shapefiles(
        area_rooftops,
        folder_path,
        f"raw {source} rooftops",
        formats=["parquet", "kml"],
    )

    ## Get rooftop stats
    area_rooftops["area_in_meters"].hist(bins=50)
    plt.xlabel("Rooftop area (m2)")
    plt.ylabel("Number of rooftops")
    plt.title(f"{source} Rooftop Area Distribution")
    plt.savefig(folder_path / "rooftop_area_distribution.png")
    plt.close()

#### Pivot to get area-per-row dataset

In [25]:
rooftop_totals_df = (
    subset_rooftops_gdf.groupby("parcel_id")
    .agg(
        total_rooftop_area_m2=("area_in_meters", "sum"),
        total_rooftop_count=("parcel_id", "size"),
    )
    .reset_index()
)

# Rename the columns to have spaces
rooftop_totals_df = rooftop_totals_df.rename(
    columns={
        "total_rooftop_area_m2": "Area Covered by Buildings (m2)",
        "total_rooftop_count": "Rooftop Count",
    }
)

In [26]:
gdf_with_stats = gdf.merge(rooftop_totals_df, on="parcel_id")

In [27]:
gdf_with_stats["Percentage Area Covered by Buildings (%)"] = gdf_with_stats["Area Covered by Buildings (m2)"] / gdf_with_stats["Parcel Area (m2)"] * 100

### Solar datasets

In [28]:
gdf_with_stats_1 = gdf_with_stats.copy()

In [29]:
solar_data_folderpath = (
    RAW_DATA_DIR
    / "solar_atlas"
    / "India_GISdata_LTAy_YearlyMonthlyTotals_GlobalSolarAtlas-v2_GEOTIFF"
)

#### Averages

In [30]:
solar_irradiation_filenames = [
    "GTI.tif",
    # "DIF.tif",
    # "DNI.tif",
    "GHI.tif",
    "PVOUT.tif",
    "TEMP.tif",
    "OPTA.tif",
    
]
solar_irradiation_column_names = [
    "Average Yearly Optimum Tilt Irradiation, GTI (kWh/m2)",
    # "Average Yearly Diffuse Irradiation, DIF (kWh/m2)",
    # "Average Yearly Direct Normal Irradiation, DNI (kWh/m2)",
    "Average Yearly Horizontal Irradiation, GHI (kWh/m2)",
    "Average Yearly Potential PV Output, PVOUT (kWh/kWp)",
    "Average Yearly Temperature (C)",
    "Average Yearly Optimum Angle (deg)",
]

In [31]:
solar_stats = []
for filename, column_name in zip(
    solar_irradiation_filenames, solar_irradiation_column_names
):
    raster = rasterio.open(solar_data_folderpath / filename)
    # Prep data
    array = raster.read(1)
    affine = raster.transform
    # get per-rooftop irradiation
    solar_stats = zonal_stats(
        gdf,
        array,
        affine=affine,
        nodata=raster.nodata,
        stats=["mean"],
        all_touched=True,
    )
    # add to rooftops dataset
    gdf[column_name] = [d["mean"] for d in solar_stats]
    # get per-colony stat
    irradiation_colony_totals = gdf.groupby("parcel_id")[
        column_name
    ].mean()

    # add to dataset
    gdf_with_stats_1[column_name] = gdf_with_stats_1["parcel_id"].map(irradiation_colony_totals)

#### Total yearly GTI, DIF, DNI, GHI

In [32]:
solar_irradiation_filenames = [
    "GTI.tif",
    # "DIF.tif",
    # "DNI.tif",
    "GHI.tif",
]
solar_irradiation_column_names = [
    "Total Yearly Optimum Tilt Irradiation, GTI (kWh)",
    # "Total Yearly Diffuse Irradiation, DIF (kWh)",
    # "Total Yearly Direct Normal Irradiation, DNI (kWh)",
    "Total Yearly Horizontal Irradiation, GHI (kWh)",
]

In [33]:
solar_stats = []
for filename, column_name in zip(
    solar_irradiation_filenames, solar_irradiation_column_names
):
    raster = rasterio.open(solar_data_folderpath / filename)
    # Prep data
    array = raster.read(1)
    affine = raster.transform
    # get per-rooftop irradiation
    solar_stats = zonal_stats(
        gdf,
        array,
        affine=affine,
        nodata=raster.nodata,
        stats=["sum"],
        all_touched=True,
    )
    # add to rooftops dataset
    gdf[column_name] = [d["sum"] for d in solar_stats]
    # get per-colony stat
    irradiation_colony_totals = gdf.groupby("parcel_id")[
        column_name
    ].sum()

    # add to dataset
    gdf_with_stats_1[column_name] = gdf_with_stats_1["parcel_id"].map(irradiation_colony_totals)

In [ ]:
gdf_with_stats_1.head()

### Slope

Source: https://bhuvan-app3.nrsc.gov.in/data/download/index.php

ISRO CartoDEM Version-3 R1, 30m resolution. The Cartosat-1 Digital Elevation Model (CartoDEM) is a National DEM developed by the Indian Space Research Organization (ISRO). It is derived from the Cartosat-1 stereo payload launched in May 2005. PDFs in folder.


In [35]:
gdf_with_stats_2 = gdf_with_stats_1.copy()

#### Load slope data

In [36]:
def get_slope_array(dem_array, cellsize=30):
    px, py = np.gradient(dem_array, cellsize)
    slope = np.sqrt(px ** 2 + py ** 2)
    return np.degrees(np.arctan(slope))

In [37]:
# Function to calculate the percentage of area with slope > 7 degrees
def calculate_slope_percentage(shape, slope, affine):
    mask = geometry_mask([shape], transform=affine, invert=True, out_shape=slope.shape)
    slope_in_shape = slope[mask]
    total_area = np.sum(mask)
    steep_area = np.sum(slope_in_shape > 7)
    return (steep_area / total_area) * 100

In [ ]:
dem_filenames = ["cdnf44a", "cdnf43i", "cdnf43j"]

for dem_filename in dem_filenames:

    path = RAW_DATA_DIR / "elevation" / f"{dem_filename}.tif"
    dem_dataset = rasterio.open(path)
    dem = dem_dataset.read(1)
    affine = dem_dataset.transform

    # get bounding box of dem
    dem_bounds = dem_dataset.bounds
    dem_shape = shapely.geometry.box(*dem_bounds)

    # mask gdf to rows that intersect this bounding box
    gdf_dem_mask = gdf_with_stats_2.intersects(dem_shape)

    # get elevation stats
    stats = zonal_stats(
        gdf_with_stats_2[gdf_dem_mask],
        dem,
        affine=affine,
        stats=["min", "max", "mean"],
        all_touched=True,
    )
    gdf_with_stats_2.loc[gdf_dem_mask, 'Elevation Average (meters)'] = [entry["mean"] for entry in stats]
    gdf_with_stats_2.loc[gdf_dem_mask, 'Elevation Mininimum (meters)'] = [entry["min"] for entry in stats]
    gdf_with_stats_2.loc[gdf_dem_mask, 'Elevation Maximum (meters)'] = [entry["max"] for entry in stats]


    # calculate slopes
    slope_array = get_slope_array(dem, cellsize=30)

    # get stats
    stats = zonal_stats(
        gdf_with_stats_2[gdf_dem_mask],
        slope_array,
        affine=affine,
        stats=["min", "max", "mean"],
        all_touched=True,
    )
    gdf_with_stats_2.loc[gdf_dem_mask, 'Slope Average (degrees)'] = [entry["mean"] for entry in stats]
    gdf_with_stats_2.loc[gdf_dem_mask, 'Slope Minimum (degrees)'] = [entry["min"] for entry in stats]
    gdf_with_stats_2.loc[gdf_dem_mask, 'Slope Maximum (degrees)'] = [entry["max"] for entry in stats]

    # get cutoff stats
    steep_area_perc = gdf_with_stats_2[gdf_dem_mask].geometry.apply(lambda shape: calculate_slope_percentage(shape, slope_array, affine))
    gdf_with_stats_2.loc[gdf_dem_mask, 'Percentage Area with Slope > 7 degrees'] = steep_area_perc

    dem_dataset.close()

In [ ]:
gdf_with_stats_2.head()

### Ground water

Source: https://indiawris.gov.in/wris/#/geoSpatialData

India Water Resource Information System > Ground Water Resource Estimation > Ground Water Resource SOD AU 2017

In [40]:
water_2017_sod_gdf = gpd.read_file(
    RAW_DATA_DIR / "water" / "Ground Water Resource SOD AU 2017.geojson"
)
# filter to MP-only
water_2017_sod_gdf = water_2017_sod_gdf[water_2017_sod_gdf["State_Name_2017"] == "MP"]

In [ ]:
water_2017_sod_gdf.plot(column="CLASS", legend=True)

In [42]:
# select columns
groundwater_columns_to_keep = [
    "CLASS",
    "Stage_of_Ground_Water__Development____",
    "Net_Annual_Ground_Water_Availability",
    "Annual_Ground_Water_Draft_Total",
    "Annual_Ground_Water_Draft_Irrigation",
    "Annual_Ground_Water_Draft_Domestic_and_industrial_uses",
    "geometry"
]
water_2017_sod_gdf = water_2017_sod_gdf[groundwater_columns_to_keep]

# rename columns
water_2017_sod_gdf = water_2017_sod_gdf.rename(
    columns={
        "CLASS": "Stage of Groundwater Development Class",
        "Stage_of_Ground_Water__Development____": "Groundwater Development Stage (%)",
        "Net_Annual_Ground_Water_Availability": "Net Annual Groundwater Availability (ham)",
        "Annual_Ground_Water_Draft_Total": "Annual Groundwater Draft Total (ham)",
        "Annual_Ground_Water_Draft_Irrigation": "Annual Groundwater Draft Irrigation (ham)",
        "Annual_Ground_Water_Draft_Domestic_and_industrial_uses": "Annual Groundwater Draft Domestic and Industrial Uses (ham)",
    }
)

In [43]:
gdf_with_stats_3 = gdf_with_stats_2.sjoin(
    water_2017_sod_gdf,
    how="left",
    predicate="intersects",
).drop(columns=["index_right"])

In [ ]:
gdf_with_stats_3

### Distance from water bodies

In [45]:
water_bodies_gdf = gpd.read_file(RAW_DATA_DIR / "water" / "DWA Waterbodies Ph1 for Madhya Pradesh.geojson")
water_bodies_gdf = water_bodies_gdf.to_crs("24378")

In [46]:
# select columns
waterbody_cols_to_keep = [
    "GmlID",
    "uuid",
    "river",
    "basin",
    "subbasin",
    "wshed",
    "waterbody_",
    "nearest_se",
    "shape_leng",
    "st_area_shape_",
    "geometry",
]
water_bodies_gdf = water_bodies_gdf[waterbody_cols_to_keep]

# rename
water_bodies_gdf = water_bodies_gdf.rename(
    columns={
        "GmlID": "Water Body GmlID",
        "uuid": "Water Body UUID",
        "river": "River",
        "basin": "Basin",
        "subbasin": "Subbasin",
        "wshed": "Watershed",
        "waterbody_": "Waterbody",
        "st_area_shape_": "Water Body Area",
        "shape_leng": "Water Body Length",
    }
)

In [47]:
# sjoin nearest waterbody
gdf_with_stats_4_geocrs = gpd.sjoin_nearest(
    gdf_with_stats_3.to_crs("24378"),
    water_bodies_gdf,
    how="left",
    max_distance=None,
    lsuffix="parcel",
    rsuffix="water_body",
    distance_col="Distance to Nearest Water Body (meters)",
).drop(columns=["index_water_body"])

### Landcover

In [48]:
path = "../data/00_raw/landcover/30N_070E_2020.tif"
src = rasterio.open(path)

In [49]:
masked_landcover_data, masked_transform = rasterio.mask.mask(src, [gdf.unary_union], crop=True)
masked_landcover_data = np.squeeze(masked_landcover_data)

In [ ]:
show(masked_landcover_data)

In [51]:
# cmap = {1.0: 'low', 2.0: 'med', 5.0: 'high'}

In [53]:
legend_df = pd.read_csv(RAW_DATA_DIR / "landcover" / "legend_processed.csv")
column_name_map = legend_df.set_index("map_value")["class_b"].to_dict()

In [ ]:
stats = zonal_stats(
    gdf,
    masked_landcover_data,
    affine=masked_transform,
    categorical=True,
    all_touched=True,
    category_map=column_name_map,
)

In [ ]:
# make list of dics into df with keys as columns and values as rows
df = pd.DataFrame(stats)
df = df.fillna(0)

# add total count column
df["total_count"] = df.sum(axis=1)
df

In [ ]:
# divide by total count to get percentage
df_perc = (df.div(df["total_count"], axis=0) * 100).round(2).drop(columns="total_count")
# add % to column names
df_perc.columns = [f"{col} %" for col in df_perc.columns]
df_perc

In [ ]:
gdf_with_stats_5_geocrs = gdf_with_stats_4_geocrs.join(df_perc)

### Distance from roads

### Weather

### Merge in admin shapes

## Save to File

In [64]:
save_shapefiles(
    gdf_with_stats_5_geocrs.to_crs(epsg=4326),
    PROCESSED_DATA_DIR,
    "land_parcels_with_stats",
    formats=["parquet", "kml", "csv"],
)

In [ ]:
# one sheet for sagar and one for dhar. Drop the unique ID.


# make a variable data sheet

